# Hidden Gems V3

En este notebook se construye una definición más estricta de `hidden gems`.

La versión anterior usaba criterios a nivel track, como `popularity < 40`, `release_year >= 2010` y `hit_score >= 80`. Sin embargo, esto no era suficiente, porque una canción con baja popularidad en un `track_id` específico podía pertenecer a un artista globalmente masivo.

Para corregir esto, se usa `tracks_enriched_artists.parquet`, una versión enriquecida del dataset que incluye variables agregadas a nivel artista:

- `max_artist_popularity`
- `max_artist_followers`
- `artist_names_ref`
- `artist_popularity_classes`
- `artist_size_classes`

La definición V3 busca canciones que sean:

- poco populares a nivel track;
- relativamente recientes;
- con alto `Hit Score`;
- asociadas a artistas no masivos;
- asociadas a artistas con popularidad moderada o baja.

Esta versión es más estricta y más defendible para la narrativa de “hidden gems”.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("HitMakerAI_HiddenGems_V3")
    .getOrCreate()
)

tracks_enriched = spark.read.parquet("../processed/tracks_enriched_artists.parquet")

tracks_enriched.printSchema()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/21 14:59:27 WARN Utils: Your hostname, debian, resolves to a loopback address: 127.0.1.1; using 192.168.100.25 instead (on interface wlp3s0)
26/05/21 14:59:27 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/21 14:59:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
[Stage 0:>                                                          (0 + 1) / 1]

root
 |-- track_id: string (nullable = true)
 |-- track_name: string (nullable = true)
 |-- artists: string (nullable = true)
 |-- id_artists: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- release_decade: integer (nullable = true)
 |-- duration_ms: integer (nullable = true)
 |-- duration_min: double (nullable = true)
 |-- explicit: integer (nullable = true)
 |-- popularity: integer (nullable = true)
 |-- popularity_class: string (nullable = true)
 |-- is_hit: integer (nullable = true)
 |-- danceability: double (nullable = true)
 |-- energy: double (nullable = true)
 |-- key: integer (nullable = true)
 |-- loudness: double (nullable = true)
 |-- mode: integer (nullable = true)
 |-- speechiness: double (nullable = true)
 |-- acousticness: double (nullable = true)
 |-- instrumentalness: double (nullable = true)
 |-- liveness: double (nullable = true)
 |-- valence: double (nullable = true)
 |-- tempo: double (nullable

## Corrección metodológica: recalcular Hit Score ponderado

El archivo `tracks_enriched_artists.parquet` fue construido a partir de `tracks_scored.parquet`, que contenía el `hit_score` original basado en promedio simple.

Para mantener consistencia con el notebook `03b_hit_score_v2`, se recalcula un `hit_score_weighted` usando los pesos derivados de la correlación absoluta con `popularity`.

A partir de esta sección, los filtros de hidden gems usarán `hit_score_weighted` como score acústico principal.

In [2]:
# Recalcular Hit Score ponderado usando pesos de V2

W = {
    "acousticness": 0.25988213907708085,
    "loudness": 0.23016439975443223,
    "energy": 0.21218715648715247,
    "instrumentalness": 0.16620344705647533,
    "danceability": 0.13156285762485917
}

tracks_enriched = (
    tracks_enriched
    .withColumn(
        "hit_score_weighted",
        F.round(
            (
                (1 - F.col("acousticness_norm")) * F.lit(W["acousticness"]) +
                F.col("loudness_norm") * F.lit(W["loudness"]) +
                F.col("energy_norm") * F.lit(W["energy"]) +
                (1 - F.col("instrumentalness_norm")) * F.lit(W["instrumentalness"]) +
                F.col("danceability_norm") * F.lit(W["danceability"])
            ) * 100,
            2
        )
    )
    .withColumn(
        "hit_score_weighted_class",
        F.when(F.col("hit_score_weighted") < 40, "bajo")
         .when(F.col("hit_score_weighted") < 70, "medio")
         .otherwise("alto")
    )
)

In [3]:
tracks_enriched.select(
    "track_name",
    "popularity",
    "hit_score",
    "hit_score_weighted",
    "danceability_norm",
    "energy_norm",
    "loudness_norm",
    "acousticness_norm",
    "instrumentalness_norm"
).orderBy(F.desc("hit_score_weighted")).show(20, truncate=False)

+-------------------------------------------+----------+---------+------------------+------------------+-----------+------------------+--------------------+---------------------+
|track_name                                 |popularity|hit_score|hit_score_weighted|danceability_norm |energy_norm|loudness_norm     |acousticness_norm   |instrumentalness_norm|
+-------------------------------------------+----------+---------+------------------+------------------+-----------+------------------+--------------------+---------------------+
|Cerol na Mão Techno                        |40        |95.48    |95.11             |0.9712092130518234|0.96       |0.8601762289651517|0.017269076305220885|1.37E-6              |
|Išvažiuosiu Prie Jūros                     |19        |95.15    |94.88             |0.9605459586265728|0.991      |0.815423347025308 |8.58433734939759E-4 |0.0085               |
|Critical Beatdown                          |36        |94.87    |94.78             |0.9381531243335467|0

In [4]:
#crear criterios base
TRACK_POPULARITY_MAX = 40
MIN_RELEASE_YEAR = 2010
MIN_HIT_SCORE = 80
MAX_ARTIST_POPULARITY = 60
MAX_ARTIST_FOLLOWERS = 500000

In [5]:
tracks_hg_flags = (
    tracks_enriched
    .withColumn(
        "is_low_track_popularity",
        F.col("popularity") < TRACK_POPULARITY_MAX
    )
    .withColumn(
        "is_recent_enough",
        F.col("release_year") >= MIN_RELEASE_YEAR
    )
    .withColumn(
        "is_high_hit_score",
        F.col("hit_score_weighted") >= MIN_HIT_SCORE
    )
    .withColumn(
        "has_artist_match",
        F.col("max_artist_popularity").isNotNull() &
        F.col("max_artist_followers").isNotNull()
    )
    .withColumn(
        "is_artist_too_popular",
        F.col("max_artist_popularity") >= MAX_ARTIST_POPULARITY
    )
    .withColumn(
        "is_artist_too_massive",
        F.col("max_artist_followers") >= MAX_ARTIST_FOLLOWERS
    )
)

In [6]:
hidden_gems_v2_like = (
    tracks_hg_flags
    .filter(
        F.col("is_low_track_popularity") &
        F.col("is_recent_enough") &
        F.col("is_high_hit_score")
    )
)

hidden_gems_v2_like.select(
    F.count("*").alias("num_candidates_track_only")
).show()

[Stage 2:==============>                                            (2 + 6) / 8]

+-------------------------+
|num_candidates_track_only|
+-------------------------+
|                    20226|
+-------------------------+



In [7]:
hidden_gems_v2_like.select(
    "track_id",
    "track_name",
    "artists",
    "release_year",
    "popularity",
    "hit_score",
    "max_artist_popularity",
    "max_artist_followers",
    "artist_size_classes"
).orderBy(F.desc("hit_score")).show(30, truncate=False, vertical=True)

-RECORD 0------------------------------------------------------------------------------------
 track_id              | 2ui62KKhQgwsUF8afLpyXZ                                              
 track_name            | Funky Cold Medina - Re-Recorded                                     
 artists               | ['Tone-Loc']                                                        
 release_year          | 2010                                                                
 popularity            | 8                                                                   
 hit_score             | 93.73                                                               
 max_artist_popularity | 52                                                                  
 max_artist_followers  | 176667.0                                                            
 artist_size_classes   | [grande]                                                            
-RECORD 1---------------------------------------------------

In [8]:
false_positive_summary = hidden_gems_v2_like.select(
    F.count("*").alias("track_only_candidates"),
    F.sum((~F.col("has_artist_match")).cast("int")).alias("without_artist_match"),
    F.sum(F.col("is_artist_too_popular").cast("int")).alias("artist_popularity_ge_60"),
    F.sum(F.col("is_artist_too_massive").cast("int")).alias("artist_followers_ge_500k"),
    F.sum(
        (
            F.col("has_artist_match") &
            (F.col("is_artist_too_popular") | F.col("is_artist_too_massive"))
        ).cast("int")
    ).alias("excluded_by_any_artist_filter")
)

false_positive_summary.show()

+---------------------+--------------------+-----------------------+------------------------+-----------------------------+
|track_only_candidates|without_artist_match|artist_popularity_ge_60|artist_followers_ge_500k|excluded_by_any_artist_filter|
+---------------------+--------------------+-----------------------+------------------------+-----------------------------+
|                20226|                1275|                   4964|                    3877|                         5156|
+---------------------+--------------------+-----------------------+------------------------+-----------------------------+



In [9]:
#ejemplos de los que serán excluidos
hidden_gems_v2_like.filter(
    F.col("is_artist_too_popular") | F.col("is_artist_too_massive")
).select(
    "track_name",
    "artists",
    "release_year",
    "popularity",
    "hit_score",
    "max_artist_popularity",
    "max_artist_followers",
    "artist_names_ref",
    "artist_size_classes"
).orderBy(F.desc("max_artist_followers")).show(
    30,
    truncate=False,
    vertical=True
)

[Stage 9:>                                                          (0 + 8) / 8]

-RECORD 0----------------------------------------------------------------
 track_name            | You Need Me, I Don't Need You                   
 artists               | ['Ed Sheeran']                                  
 release_year          | 2011                                            
 popularity            | 37                                              
 hit_score             | 83.62                                           
 max_artist_popularity | 92                                              
 max_artist_followers  | 7.8900234E7                                     
 artist_names_ref      | [Ed Sheeran]                                    
 artist_size_classes   | [masivo]                                        
-RECORD 1----------------------------------------------------------------
 track_name            | Love Me Harder                                  
 artists               | ['Ariana Grande', 'The Weeknd']                 
 release_year          | 2014         

In [10]:
hidden_gems_v3_candidates = (
    hidden_gems_v2_like
    .filter(
        F.col("has_artist_match") &
        (~F.col("is_artist_too_popular")) &
        (~F.col("is_artist_too_massive"))
    )
)

hidden_gems_v3_candidates.select(
    F.count("*").alias("num_hidden_gems_v3_candidates")
).show()

[Stage 10:==============>                                           (2 + 6) / 8]

+-----------------------------+
|num_hidden_gems_v3_candidates|
+-----------------------------+
|                        13795|
+-----------------------------+



In [11]:
#candidatos
hidden_gems_v3_candidates.select(
    "track_id",
    "track_name",
    "artists",
    "release_year",
    "popularity",
    "hit_score",
    "max_artist_popularity",
    "max_artist_followers",
    "artist_size_classes"
).orderBy(F.desc("hit_score")).show(30, truncate=False, vertical=True)

[Stage 13:=======>                                                  (1 + 7) / 8]

-RECORD 0------------------------------------------------------------------------------------
 track_id              | 2ui62KKhQgwsUF8afLpyXZ                                              
 track_name            | Funky Cold Medina - Re-Recorded                                     
 artists               | ['Tone-Loc']                                                        
 release_year          | 2010                                                                
 popularity            | 8                                                                   
 hit_score             | 93.73                                                               
 max_artist_popularity | 52                                                                  
 max_artist_followers  | 176667.0                                                            
 artist_size_classes   | [grande]                                                            
-RECORD 1---------------------------------------------------

In [12]:
hidden_gems_v3_candidates = (
    hidden_gems_v3_candidates
    .withColumn(
        "track_name_norm",
        F.lower(
            F.trim(
                F.regexp_replace(F.col("track_name"), r"[^\p{L}\p{N}]+", " ")
            )
        )
    )
    .withColumn(
        "artist_key",
        F.concat_ws("_", F.array_sort(F.col("artist_ids")))
    )
)

In [13]:
#componente temporal
import math
from pyspark.sql import functions as F
from pyspark.sql.window import Window

max_year = tracks_enriched.select(
    F.max("release_year").alias("max_year")
).first()["max_year"]

half_life = 10
lambda_decay = math.log(2) / half_life

hidden_gems_v3_candidates = (
    hidden_gems_v3_candidates
    .withColumn(
        "song_age",
        F.when(
            F.col("release_year").isNotNull(),
            F.lit(max_year) - F.col("release_year")
        ).otherwise(F.lit(None).cast("int"))
    )
    .withColumn(
        "recency_weight",
        F.when(
            F.col("song_age").isNotNull(),
            F.exp(-F.lit(lambda_decay) * F.col("song_age"))
        ).otherwise(F.lit(0.0))
    )
    .withColumn(
        "recency_score",
        F.round(F.col("recency_weight") * 100, 2)
    )
    .withColumn(
    "hidden_gem_score",
    F.round(
        0.80 * F.col("hit_score_weighted") +
        0.20 * F.col("recency_score"),
        2
        )
    )
)

In [14]:
hidden_gems_v3_candidates.select(
    "track_name",
    "release_year",
    "song_age",
    "popularity",
    "hit_score",
    "recency_score",
    "hidden_gem_score",
    "max_artist_popularity",
    "max_artist_followers"
).orderBy(F.desc("hidden_gem_score")).show(20, truncate=False, vertical=True)

-RECORD 0------------------------------------------------------
 track_name            | Predestination                        
 release_year          | 2020                                  
 song_age              | 1                                     
 popularity            | 39                                    
 hit_score             | 92.03                                 
 recency_score         | 93.3                                  
 hidden_gem_score      | 92.77                                 
 max_artist_popularity | 44                                    
 max_artist_followers  | 112.0                                 
-RECORD 1------------------------------------------------------
 track_name            | Linnud                                
 release_year          | 2020                                  
 song_age              | 1                                     
 popularity            | 36                                    
 hit_score             | 91.37          

## Hidden Gems v3 acústica
¿Qué canciones recientes, poco populares y de artistas no masivos tienen mayor perfil acústico de hit?

In [18]:
dedup_window_acoustic = Window.partitionBy(
    "track_name_norm",
    "artist_key"
).orderBy(
    F.desc("hit_score_weighted"),
    F.desc("release_year"),
    F.desc("popularity")
)

hidden_gems_v3_acoustic = (
    hidden_gems_v3_candidates
    .withColumn("rn", F.row_number().over(dedup_window_acoustic))
    .filter(F.col("rn") == 1)
    .drop("rn")
    .select(
        "track_id",
        "track_name",
        "artists",
        "artist_names_ref",
        "release_year",
        "release_decade",
        "song_age",
        "popularity",
        "popularity_class",
        "hit_score",
        "hit_score_weighted",
        "recency_score",
        "hidden_gem_score",
        "hit_score_weighted_class",
        "max_artist_popularity",
        "max_artist_followers",
        "artist_popularity_classes",
        "artist_size_classes",
        "danceability",
        "energy",
        "loudness",
        "acousticness",
        "instrumentalness",
        "valence",
        "tempo",
        "duration_min"
    )
    .orderBy(F.desc("hit_score_weighted"))
)

hidden_gems_v3_acoustic.show(30, truncate=False, vertical=True)

[Stage 45:>                                                         (0 + 8) / 8]

-RECORD 0----------------------------------------------------------------------------------------
 track_id                  | 0Jlyt9eybWfVn4PzpEhQGI                                              
 track_name                | Those - Mix Cut                                                     
 artists                   | ['D.O.D']                                                           
 artist_names_ref          | [D.O.D]                                                             
 release_year              | 2015                                                                
 release_decade            | 2010                                                                
 song_age                  | 6                                                                   
 popularity                | 2                                                                   
 popularity_class          | baja                                                                
 hit_score          

## Hidden Gems con recencia 

In [19]:
dedup_window_recency = Window.partitionBy(
    "track_name_norm",
    "artist_key"
).orderBy(
    F.desc("hidden_gem_score"),
    F.desc("hit_score_weighted"),
    F.desc("release_year"),
    F.desc("popularity")
)

hidden_gems_v3_recency = (
    hidden_gems_v3_candidates
    .withColumn("rn", F.row_number().over(dedup_window_recency))
    .filter(F.col("rn") == 1)
    .drop("rn")
    .select(
        "track_id",
        "track_name",
        "artists",
        "artist_names_ref",
        "release_year",
        "release_decade",
        "song_age",
        "popularity",
        "popularity_class",
        "hit_score",
        "hit_score_weighted",
        "recency_score",
        "hidden_gem_score",
        "hit_score_weighted_class",
        "max_artist_popularity",
        "max_artist_followers",
        "artist_popularity_classes",
        "artist_size_classes",
        "danceability",
        "energy",
        "loudness",
        "acousticness",
        "instrumentalness",
        "valence",
        "tempo",
        "duration_min"
    )
    .orderBy(F.desc("hidden_gem_score"))
)

hidden_gems_v3_recency.show(30, truncate=False, vertical=True)

[Stage 48:=====================>                                    (3 + 5) / 8]

-RECORD 0----------------------------------------------------------------------------------------
 track_id                  | 6rGaZ6NAMkXrTBN0H72w1D                                              
 track_name                | Predestination                                                      
 artists                   | ['Ryan']                                                            
 artist_names_ref          | [Ryan]                                                              
 release_year              | 2020                                                                
 release_decade            | 2020                                                                
 song_age                  | 1                                                                   
 popularity                | 39                                                                  
 popularity_class          | baja                                                                
 hit_score          

In [20]:
#comparamos conteos
comparison_summary = spark.createDataFrame(
    [
        ("v2_like_track_only_weighted", hidden_gems_v2_like.count()),
        ("v3_candidates_artist_filtered", hidden_gems_v3_candidates.count()),
        ("v3_acoustic_after_dedup", hidden_gems_v3_acoustic.count()),
        ("v3_recency_after_dedup", hidden_gems_v3_recency.count())
    ],
    ["stage", "num_tracks"]
)

comparison_summary.show(truncate=False)

+-----------------------------+----------+
|stage                        |num_tracks|
+-----------------------------+----------+
|v2_like_track_only_weighted  |20226     |
|v3_candidates_artist_filtered|13795     |
|v3_acoustic_after_dedup      |13176     |
|v3_recency_after_dedup       |13176     |
+-----------------------------+----------+



### Comparación final de etapas con Hit Score ponderado

Después de corregir el notebook para usar `hit_score_weighted`, se repitió el proceso de construcción de `hidden_gems_v3`.

Primero se generó una lista de candidatos usando únicamente criterios a nivel track:

- `popularity < 40`
- `release_year >= 2010`
- `hit_score_weighted >= 80`

Esta primera etapa produjo 20,226 candidatos.

Después se aplicaron filtros a nivel artista:

- el track debe tener información de artista;
- `max_artist_popularity < 60`;
- `max_artist_followers < 500000`.

Con estos filtros, la lista se redujo a 13,795 candidatos.

Finalmente, se aplicó deduplicación por nombre normalizado de canción y conjunto de artistas, dejando 13,176 canciones finales.

Se construyeron dos rankings:

- `hidden_gems_v3_acoustic`, ordenado por `hit_score_weighted`;
- `hidden_gems_v3_recency`, ordenado por `hidden_gem_score`.

Ambas listas tienen el mismo número de canciones porque parten del mismo conjunto filtrado y deduplicado. La diferencia está en el criterio de ordenamiento.

## Guardamos

In [21]:
hidden_gems_v3_acoustic.write.mode("overwrite").parquet(
    "../processed/hidden_gems_v3_acoustic.parquet"
)

hidden_gems_v3_recency.write.mode("overwrite").parquet(
    "../processed/hidden_gems_v3_recency.parquet"
)

comparison_summary.write.mode("overwrite").parquet(
    "../processed/diagnostics/hidden_gems_v3/comparison_summary.parquet"
)

In [22]:
#Lista para presentación
version_pattern = r"(?i)(re-recorded|re recorded|karaoke|tribute|cover|live|remix|radio edit|workout|version|mix cut)"

hidden_gems_v3_presentation = (
    hidden_gems_v3_recency
    .filter(~F.col("track_name").rlike(version_pattern))
    .orderBy(F.desc("hidden_gem_score"))
)

hidden_gems_v3_presentation.show(30, truncate=False, vertical=True)

[Stage 89:=====================>                                    (3 + 5) / 8]

-RECORD 0----------------------------------------------------------------------------------------
 track_id                  | 6rGaZ6NAMkXrTBN0H72w1D                                              
 track_name                | Predestination                                                      
 artists                   | ['Ryan']                                                            
 artist_names_ref          | [Ryan]                                                              
 release_year              | 2020                                                                
 release_decade            | 2020                                                                
 song_age                  | 1                                                                   
 popularity                | 39                                                                  
 popularity_class          | baja                                                                
 hit_score          

In [23]:
hidden_gems_v3_presentation.write.mode("overwrite").parquet(
    "../processed/hidden_gems_v3_presentation.parquet"
)

## Conclusión de Hidden Gems V3

En este notebook se construyó una versión más estricta de hidden gems.

Primero se generaron candidatos usando criterios a nivel track: baja popularidad, lanzamiento reciente y alto `hit_score_weighted`. Después se incorporaron filtros a nivel artista usando `max_artist_popularity` y `max_artist_followers`.

Esto corrige una limitación detectada en versiones anteriores: un track podía tener baja popularidad local, pero pertenecer a un artista globalmente masivo.

La versión final usa:

- `popularity < 40`
- `release_year >= 2010`
- `hit_score_weighted >= 80`
- `max_artist_popularity < 60`
- `max_artist_followers < 500000`

Además, se construyeron dos rankings:

- `hidden_gems_v3_acoustic`: ordenado por `hit_score_weighted`.
- `hidden_gems_v3_recency`: ordenado por `hidden_gem_score`, una combinación de 80% `hit_score_weighted` y 20% `recency_score`.

Ambas listas tienen el mismo número de canciones porque parten del mismo conjunto filtrado y deduplicado. La diferencia está en el criterio de ordenamiento.